#Лабораторна робота 2
###Варінат 2: Impact of AI on Students
Виконала: Слота Аліна

In [1]:
import pandas as pd
import numpy as np
import sqlite3

# Завантажуємо дані з бази
conn = sqlite3.connect('students.db')
data = pd.read_sql("SELECT * FROM student_impact", conn)
conn.close()

data.head()

,Student_ID,Major_Category,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Primary_Use_Case,Prompt_Engineering_Skill,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Institutional_Policy,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score,Burnout_Risk_Level,timestamp
0,100001,Humanities,Senior,2.418,23.31,Copywriting/Drafting,Beginner,1,1,8.13,5,Allowed_With_Citation,6,2.393,86.44,High,2025-10-22 00:01:55
1,100002,Medical,Junior,3.821,1.12,Ideation,Advanced,5,0,16.65,3,Allowed_With_Citation,9,3.696,69.39,Low,2025-07-07 22:50:00
2,100003,Business,Freshman,3.398,21.26,Summarizing_Reading,Beginner,2,0,10.35,5,Strict_Ban,9,3.499,73.93,Medium,2025-11-03 21:00:15
3,100004,Business,Senior,3.789,1.82,Copywriting/Drafting,Intermediate,4,0,15.23,2,Allowed_With_Citation,2,4.000,63.58,Medium,2025-02-10 07:05:30
4,100005,STEM,Sophomore,3.635,9.29,Debugging/Troubleshooting,Advanced,4,0,12.55,4,Allowed_With_Citation,4,3.798,100.00,Medium,2025-10-02 05:32:09


## Обробка відстуніх значень та аномалій

In [2]:
missing = data.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Пропущених значень немає")

Пропущених значень немає


In [3]:
import numpy as np
import matplotlib.pyplot as plt

# підрахунок викидів з ігноруванням бінарних та категоріальних колонок
def count_outliers(df):
    outliers_info = {}
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        if df[col].nunique() < 10:
            continue
            
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR_val = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR_val
        upper_bound = Q3 + 1.5 * IQR_val
        
        n_outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
        if n_outliers > 0:
            outliers_info[col] = n_outliers
            
    return outliers_info

outliers_dict = count_outliers(data)

if not outliers_dict:
    print("Аномальних значень (викидів) не виявлено.")
else:
    print("Знайдено викиди:")
    for col, count in outliers_dict.items():
        print(f"- {col}: {count} аномалій")

Знайдено викиди:
- Pre_Semester_GPA: 328 аномалій
- Weekly_GenAI_Hours: 2583 аномалій
- Traditional_Study_Hours: 161 аномалій
- Perceived_AI_Dependency: 190 аномалій
- Post_Semester_GPA: 346 аномалій
- Skill_Retention_Score: 216 аномалій


Перевірка показала, що жодна з 17 колонок не містить NaN та аномальних значень. Щоб застосувати методи інженерії ознак, потрібно додати штучні пропуски.

In [4]:
np.random.seed(42) # для відтворюваності результатів
n_rows = len(data)

# Штучні пропуски
mask_mean = np.random.rand(n_rows) < 0.05
data.loc[mask_mean, 'Pre_Semester_GPA'] = np.nan

mask_arbitrary = np.random.rand(n_rows) < 0.05
data.loc[mask_arbitrary, 'Post_Semester_GPA'] = np.nan

mask_indicator = np.random.rand(n_rows) < 0.04
data.loc[mask_indicator, 'Anxiety_Level_During_Exams'] = np.nan

mask_mode = np.random.rand(n_rows) < 0.05
data.loc[mask_mode, 'Primary_Use_Case'] = np.nan

mask_cat_missing = np.random.rand(n_rows) < 0.05
data.loc[mask_cat_missing, 'Prompt_Engineering_Skill'] = np.nan


# Штучні аномалії

mask_cap = np.random.rand(n_rows) < 0.02
data.loc[mask_cap, 'Weekly_GenAI_Hours'] = data['Weekly_GenAI_Hours'] * 10 + 100

mask_trim = np.random.rand(n_rows) < 0.02
data.loc[mask_trim, 'Traditional_Study_Hours'] = -100

mask_discretize = np.random.rand(n_rows) < 0.02
data.loc[mask_discretize, 'Tool_Diversity'] = 999

In [5]:
missing = data.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Пропущених значень немає")

Pre_Semester_GPA              2478
Primary_Use_Case              2469
Prompt_Engineering_Skill      2468
Anxiety_Level_During_Exams    1929
Post_Semester_GPA             2481
dtype: int64


З'явились пропущені значення.

### Обробка пропущених значень

In [6]:
# заміна медіаною
median_gpa = data['Pre_Semester_GPA'].median()
data['Pre_Semester_GPA'] = data['Pre_Semester_GPA'].fillna(median_gpa)

# заміна значенням з кінця розподілу (для числових даних, що відсутні не випадково)
IQR = data['Post_Semester_GPA'].quantile(0.75) - data['Post_Semester_GPA'].quantile(0.25)
extreme_value = data['Post_Semester_GPA'].quantile(0.75) + 3 * IQR
data['Post_Semester_GPA'] = data['Post_Semester_GPA'].fillna(extreme_value)

# додавання змінної-індикатора + заміна медіаною
data['Anxiety_Level_During_Exams_is_NA'] = np.where(data['Anxiety_Level_During_Exams'].isnull(), 1, 0) # 1 - був пропуск, 0 - не було
data['Anxiety_Level_During_Exams'] = data['Anxiety_Level_During_Exams'].fillna(data['Anxiety_Level_During_Exams'].median())

# заміна на найчастішу категорію
mode_usecase = data['Primary_Use_Case'].mode()[0]
data['Primary_Use_Case'] = data['Primary_Use_Case'].fillna(mode_usecase)

# заміна категорією 'Missing'
data['Prompt_Engineering_Skill'] = data['Prompt_Engineering_Skill'].fillna('Missing')

In [7]:
missing = data.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Пропущених значень немає")

Пропущених значень немає


Пропущені значення успішно опрацьовані.

### Обробка аномальних значень

In [8]:
import numpy as np

# підрахунок викидів з ігноруванням бінарних та категоріальних колонок
def count_outliers(df):
    outliers_info = {}
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        if df[col].nunique() < 10:
            continue
            
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR_val = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR_val
        upper_bound = Q3 + 1.5 * IQR_val
        
        n_outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
        if n_outliers > 0:
            outliers_info[col] = n_outliers
            
    return outliers_info

outliers_dict = count_outliers(data)

if not outliers_dict:
    print("Аномальних значень (викидів) не виявлено.")
else:
    print("Знайдено викиди:")
    for col, count in outliers_dict.items():
        print(f"- {col}: {count} аномалій")

Знайдено викиди:
- Pre_Semester_GPA: 489 аномалій
- Weekly_GenAI_Hours: 3190 аномалій
- Traditional_Study_Hours: 1120 аномалій
- Perceived_AI_Dependency: 190 аномалій
- Post_Semester_GPA: 2776 аномалій
- Skill_Retention_Score: 216 аномалій


З'явились аномальні значення.

In [9]:
# видалення неможливих від'ємних значень
data = data[data['Traditional_Study_Hours'] >= 0].copy()

# заміна медіаною
if 'Tool_Diversity' in data.columns:
    data['Tool_Diversity'] = data['Tool_Diversity'].replace(999, np.nan)
    data['Tool_Diversity'] = data['Tool_Diversity'].fillna(data['Tool_Diversity'].median())

# цензурування через IQR
def cap_outliers_fixed(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR_val = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR_val
    upper_bound = Q3 + 1.5 * IQR_val
    
    n_outliers = ((df[column] < lower_bound) | (df[column] > upper_bound)).sum()
    df[column] = df[column].clip(lower=lower_bound, upper=upper_bound)
    return n_outliers, lower_bound, upper_bound

# Список колонок з викидами
columns_to_cap = [
    'Weekly_GenAI_Hours',
    'Pre_Semester_GPA', 
    'Traditional_Study_Hours', 
    'Perceived_AI_Dependency', 
    'Post_Semester_GPA', 
    'Skill_Retention_Score'
]

print("Результати обробки аномалій:")
for col in columns_to_cap:
    if col in data.columns:
        n_out, lb, ub = cap_outliers_fixed(data, col)
        print(f"{col}: Оброблено {n_out} викидів (межі: {lb:.2f}, {ub:.2f})")

Результати обробки аномалій:
Weekly_GenAI_Hours: Оброблено 3144 викидів (межі: -12.24, 26.96)
Pre_Semester_GPA: Оброблено 480 викидів (межі: 1.89, 4.48)
Traditional_Study_Hours: Оброблено 155 викидів (межі: -3.19, 25.45)
Perceived_AI_Dependency: Оброблено 186 викидів (межі: -2.50, 9.50)
Post_Semester_GPA: Оброблено 2721 викидів (межі: 1.92, 4.94)
Skill_Retention_Score: Оброблено 214 викидів (межі: 39.27, 112.71)


In [10]:
import numpy as np
import matplotlib.pyplot as plt

# підрахунок викидів з ігноруванням бінарних та категоріальних колонок
def count_outliers(df):
    outliers_info = {}
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        if df[col].nunique() < 10:
            continue
            
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR_val = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR_val
        upper_bound = Q3 + 1.5 * IQR_val
        
        n_outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
        if n_outliers > 0:
            outliers_info[col] = n_outliers
            
    return outliers_info

outliers_dict = count_outliers(data)

if not outliers_dict:
    print("Аномальних значень (викидів) не виявлено.")
else:
    print("Знайдено викиди:")
    for col, count in outliers_dict.items():
        print(f"- {col}: {count} аномалій")

Аномальних значень (викидів) не виявлено.


### Кодування категоріальних ознак
Year_of_Study, Burnout_Risk_Level - ordinal encoding, оскільки мають природній порядок (Low < Medium < High);
Major_Category, Institutional_Policy, Primary_Use_Case, Prompt_Engineering_Skill - метод One Hot encoding у k-1 змінних. 

In [11]:
data_encoded = data.copy()
year_ordered = {'Freshman': 0, 'Sophomore': 1, 'Junior': 2, 'Senior': 3, 'Graduate': 4}

data_encoded['Year_of_Study'] = data_encoded['Year_of_Study'].map(year_ordered) 

burnout_ordered = {'Low': 0, 'Medium': 1, 'High': 2}
data_encoded['Burnout_Risk_Level'] = data_encoded['Burnout_Risk_Level'].map(burnout_ordered)

nominal_columns = ['Major_Category', 'Institutional_Policy', 'Primary_Use_Case', 'Prompt_Engineering_Skill']
data_encoded = pd.get_dummies(data_encoded, columns=nominal_columns, drop_first=True)

print(data_encoded.shape)
data_encoded.head()


(49019, 27)


,Student_ID,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Anxiety_Level_During_Exams,Post_Semester_GPA,...,Major_Category_STEM,Institutional_Policy_Allowed_With_Citation,Institutional_Policy_Strict_Ban,Primary_Use_Case_Debugging/Troubleshooting,Primary_Use_Case_Direct_Answer_Generation,Primary_Use_Case_Ideation,Primary_Use_Case_Summarizing_Reading,Prompt_Engineering_Skill_Beginner,Prompt_Engineering_Skill_Intermediate,Prompt_Engineering_Skill_Missing
0,100001,3,2.418,23.31,1.0,1,8.13,5.0,6.0,2.393,...,False,True,False,False,False,False,False,True,False,False
1,100002,2,3.821,1.12,5.0,0,16.65,3.0,9.0,3.696,...,False,True,False,False,False,True,False,False,False,False
2,100003,0,3.398,21.26,2.0,0,10.35,5.0,9.0,3.499,...,False,False,True,False,False,False,True,True,False,False
3,100004,3,3.789,1.82,4.0,0,15.23,2.0,2.0,4.000,...,False,True,False,False,False,False,False,False,True,False
4,100005,1,3.635,9.29,4.0,0,12.55,4.0,4.0,3.798,...,True,True,False,True,False,False,False,False,False,False


## Маштабування

In [12]:
from sklearn.preprocessing import StandardScaler
import joblib

numeric_features = ['Pre_Semester_GPA', 'Weekly_GenAI_Hours', 'Tool_Diversity',
                     'Traditional_Study_Hours', 'Perceived_AI_Dependency',
                     'Anxiety_Level_During_Exams', 'Post_Semester_GPA', 
                     'Skill_Retention_Score']

scaler = StandardScaler()
data_encoded[numeric_features] = scaler.fit_transform(data_encoded[numeric_features])
joblib.dump(scaler, 'my_standard_scaler.pkl')

data_encoded.head()

,Student_ID,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Anxiety_Level_During_Exams,Post_Semester_GPA,...,Major_Category_STEM,Institutional_Policy_Allowed_With_Citation,Institutional_Policy_Strict_Ban,Primary_Use_Case_Debugging/Troubleshooting,Primary_Use_Case_Direct_Answer_Generation,Primary_Use_Case_Ideation,Primary_Use_Case_Summarizing_Reading,Prompt_Engineering_Skill_Beginner,Prompt_Engineering_Skill_Intermediate,Prompt_Engineering_Skill_Missing
0,100001,3,-1.583543,1.911410,-1.533307,1,-0.596956,0.824215,0.828608,-1.756430,...,False,True,False,False,False,False,False,True,False,False
1,100002,2,1.447696,-0.953347,1.865805,0,1.059820,-0.277653,2.256391,0.445726,...,False,True,False,False,False,True,False,False,False,False
2,100003,0,0.533787,1.646752,-0.683529,0,-0.165261,0.824215,2.256391,0.112783,...,False,False,True,False,False,False,True,True,False,False
3,100004,3,1.378559,-0.862976,1.016027,0,0.783690,-0.828587,-1.075102,0.959506,...,False,True,False,False,False,False,False,False,True,False
4,100005,1,1.045836,0.101411,1.016027,0,0.262545,0.273281,-0.123247,0.618113,...,True,True,False,True,False,False,False,False,False,False


Використано метод стандартизаціїї, оскільки він не настільки чутливий до вкидів, як MinmaxScaling

##Звіт про очищення даних:
Оскільки початковий датасет був чистим, для випробування методів інженерії ознак у нього було штучно додано пропуски та аномалії.

Обробка пропущених значень
Пропуски у числових ознаках оброблено заміною на медіану, яка є стійкою до викидів, та значенням з кінця розподілу, що дозволяє виділити змістовність пропуску (для даних типу MNAR). Також було створено змінну-індикатор для фіксації факту відсутності даних. У категоріальних ознаках пропуски успішно заповнено найчастішою категорією (модою) та виділено в окрему групу «Missing» для збереження цілісності даних.   Опрацювання аномалій

Аномальні значення опрацьовано комбінованим підходом. Логічно неможливі дані (від'ємні години) повністю видалено з датасету методом Trimming для очищення від явного шуму. Інші екстремальні відхилення оброблено цензуруванням за правилом IQR. Це дозволило зберегти спостереження для моделі, обмеживши їх безпечними межами і мінімізувавши шкоду для лінійних алгоритмів.   Кодування та масштабування

Категоріальні ознаки з природною ієрархією закодовано методом Ordinal Encoding, а номінальні — через One-Hot Encoding у K-1 змінних, що ефективно усуває проблему мультиколінеарності. Для приведення числових ознак до єдиного масштабу використано метод стандартизації (StandardScaler), який центрує значення та є надійнішим за наявності залишкових викидів.   

Результат: датасет успішно очищено та підготовлено до етапу моделювання. Усі штучні пропуски заповнено відповідними статистичними методами, вплив аномалій нейтралізовано, категоріальні ознаки закодовано, а числові — стандартизовано.


## Обгрунтування методів інженерії даних 
Вибір методів підготовки даних базувався на властивостях датасету (включно зі штучно згенерованими шумами) та вимогах майбутньої моделі класифікації Burnout_Risk_Level.

Обробка пропусків: Заміну медіаною для числових ознак обрано через її стійкість до викидів порівняно із середнім значенням. Для збереження прогностичної цінності факту відсутності інформації (MNAR) додатково використано Missing Indicator та заміну значенням з кінця розподілу. Для категоріальних змінних використано заміну модою та створення нової категорії «Missing», що дозволяє заповнити пропуски без введення хибних припущень у дані.

Обробка аномалій: Метод видалення застосовано виключно до логічно неможливих значень (від'ємні години), які є явними помилками. Для інших екстремальних відхилень обрано цензурування замість видалення, оскільки високі показники (наприклад, у Weekly_GenAI_Hours) можуть корелювати з ризиком вигорання і несуть змістовну інформацію. Цензурування дозволило зберегти ці записи, мінімізувавши їхній негативний вплив на масштаб.   

Кодування категоріальних ознак: Ordinal Encoding обрано для змінних із чіткою природною ієрархією (Year_of_Study, Burnout_Risk_Level), оскільки збереження цього порядку є критично важливим для логіки моделі. One-Hot Encoding у K-1 змінних використано для номінальних ознак (Major_Category та ін.), щоб запобігти хибному введенню штучної ієрархії та уникнути проблеми мультиколінеарності.   

Масштабування: StandardScaler обрано через необхідність зведення ознак до єдиного масштабу для коректної роботи алгоритмів. На відміну від MinMax-масштабування, метод стандартизації зберігає форму оригінального розподілу та є більш надійним за наявності залишкових викидів.

## Збереження підготовлених даних у базу даних

In [13]:
conn = sqlite3.connect('students.db')

data_encoded.to_sql('student_impact_encoded', conn, if_exists='replace', index=False)

conn.close()
print("Таблицю student_impact_encoded успішно створено")

Таблицю student_impact_encoded успішно створено


In [14]:
conn = sqlite3.connect('students.db')

check = pd.read_sql("""
    SELECT a.Student_ID, a.Burnout_Risk_Level, b.Burnout_Risk_Level
    FROM student_impact a
    JOIN student_impact_encoded b ON a.Student_ID = b.Student_ID
    LIMIT 5
""", conn)

conn.close()
check

,Student_ID,Burnout_Risk_Level,Burnout_Risk_Level
0,100001,High,2
1,100002,Low,0
2,100003,Medium,1
3,100004,Medium,1
4,100005,Medium,1


Встановлено зв`язок 1:1 з початковим датасетом по StidentID - кожному запису в початковій таблиці відповідає 
рівно один запис у новій таблиці з тим самим ідентифікатором студента.

### Висновок:
Під час виконання другої лабораторної роботи я навчилася застосовувати повний цикл підготовки даних для алгоритмів машинного навчання. Я освоїла різні стратегії обробки пропущених значень (заміна медіаною, модою, значеннями з кінця розподілу та створення додаткових індикаторів), а також навчилася ефективно працювати з аномаліями. Зокрема, я навчилася відрізняти логічні помилки, які потребують повного видалення , від змістовних екстремальних відхилень, які варто обробляти методом цензурування за правилом IQR, щоб не втрачати важливу інформацію. Крім того, я зрозуміла, наскільки важливо підбирати методи кодування індивідуально: я застосувала Ordinal Encoding для збереження логічної ієрархії в категоріях та One-Hot Encoding для номінальних ознак. Завершивши підготовку стандартизацією числових ознак, я отримала повністю чистий датасет, придатний для тренування алгоритмів, і закріпила навички роботи з базами даних, успішно експортувавши результати в SQLite з налаштуванням зв'язку 1:1 із початковою таблицею сирих даних.